In [1]:
import pandas as pd
from neo4j import GraphDatabase
from pathlib import Path
from getpass import getpass

In [2]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

roads_df = pd.read_csv(
    project_folder / "data" / "outputs" / "roads.csv",
    dtype={"id": str}
)

print("Available roads:", len(roads_df))
print(roads_df[["id", "roadName"]].head())

Available roads: 794
              id       roadName
0    -1203892148    -1203892148
1    -1246284225    -1246284225
2  -1246284226#0  -1246284226#0
3  -1246284226#1  -1246284226#1
4    -1246285590    -1246285590


In [4]:
if len(roads_df) < 3:
    raise ValueError("At least three roads are required.")

selected = roads_df.sample(n=3, random_state=42)

incidents = [
    {
        "id": "SUMO_Accident_1",
        "incidentType": "Accident",
        "severity": 4,
        "startTime": "2026-11-03T10:02:00",
        "endTime": "2026-11-03T10:07:00",
        "roadId": "SUMO_" + selected.iloc[0]["id"],
        "source": "Simulated"
    },
    {
        "id": "SUMO_Breakdown_1",
        "incidentType": "Breakdown",
        "severity": 2,
        "startTime": "2026-11-03T10:04:00",
        "endTime": "2026-11-03T10:09:00",
        "roadId": "SUMO_" + selected.iloc[1]["id"],
        "source": "Simulated"
    },
    {
        "id": "SUMO_Roadwork_1",
        "incidentType": "Roadwork",
        "severity": 3,
        "startTime": "2026-11-03T10:00:00",
        "endTime": "2026-11-03T10:15:00",
        "roadId": "SUMO_" + selected.iloc[2]["id"],
        "source": "Simulated"
    }
]

incidents_df = pd.DataFrame(incidents)

print(incidents_df[["id", "incidentType", "severity", "roadId"]])

                 id incidentType  severity             roadId
0   SUMO_Accident_1     Accident         4     SUMO_374964887
1  SUMO_Breakdown_1    Breakdown         2     SUMO_532236701
2   SUMO_Roadwork_1     Roadwork         3  SUMO_-170204672#5


In [5]:
output = project_folder / "data" / "outputs" / "incidents.csv"

incidents_df.to_csv(output, index=False)

print("Incident dataset saved successfully!")

Incident dataset saved successfully!


In [6]:
uri = "bolt://localhost:7687"
username = "neo4j"
password = getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(uri, auth=(username, password))

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [7]:
incident_data = incidents_df.to_dict("records")

query = """
UNWIND $incidents AS row

MATCH (r:Road {id: row.roadId})

MERGE (i:Incident {id: row.id})

SET i.incidentType = row.incidentType,
    i.severity = row.severity,
    i.startTime = row.startTime,
    i.endTime = row.endTime,
    i.source = row.source

MERGE (i)-[:LOCATED_AT]->(r)
MERGE (i)-[:BLOCKS]->(r)

SET r.status = "Blocked"
"""

with driver.session(database="neo4j") as session:
    session.run(query, incidents=incident_data).consume()

print("Incident Graph created successfully!")

Incident Graph created successfully!


In [8]:
driver.close()

print("Neo4j connection closed!")

Neo4j connection closed!
